# Demo: GOES RGB + ERA5-Land Cloud Mask

This notebook takes a user-defined lat/lon box, date range, and UTC time-of-day range, then runs the full workflow: download GOES, orthorectify, build Zarr, build RGB, apply the ERA5-Land temperature-bin cloud mask, and plot RGB next to the mask.

**Important:** this RGB cloud-mask method is intended for daylight imagery only. Choose UTC hours when your full domain is sunlit. For the default Colorado demo on `2020-06-30`, `18-22 UTC` is a compact daylight window.

## 1. Set Your Domain, Dates, And Daylight Hours

Edit these values first. Longitudes west of Greenwich should be negative.

In [ ]:
from pathlib import Path

# Default test case: Colorado domain, June 30, 2020
DOMAIN = "colorado"
GOES = "goes16"
START_DATE = "2020-06-30"
END_DATE = "2020-06-30"

# Bounding box: lon_min, lat_min, lon_max, lat_max
LON_MIN = -109.0
LAT_MIN = 37.0
LON_MAX = -104.0
LAT_MAX = 41.0

# Daylight UTC window. The mask is not designed for nighttime imagery.
GOES_HOURS = "18-22"
START_HOUR_UTC = 18
END_HOUR_UTC = 22

# Keep outputs somewhere with enough space.
BASE_DIR = Path("./demo_output/colorado")

# Set False to preview the commands without downloading/processing data.
RUN_WORKFLOW = True
OVERWRITE_MASK = True

# Time shown in the final plot. The nearest available timestep is used.
PLOT_TIME_UTC = "2020-06-30 20:00"

## 2. Build The Workflow Config

In [ ]:
import sys
from pathlib import Path

REPO_DIR = Path.cwd()
if not (REPO_DIR / "scripts").exists() and (REPO_DIR.parent / "scripts").exists():
    REPO_DIR = REPO_DIR.parent
SCRIPTS_DIR = REPO_DIR / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import workflow as wf

config = wf.WorkflowConfig(
    domain=DOMAIN,
    goes=GOES,
    start_date=START_DATE,
    end_date=END_DATE,
    lon_min=LON_MIN,
    lat_min=LAT_MIN,
    lon_max=LON_MAX,
    lat_max=LAT_MAX,
    base_dir=BASE_DIR,
    goes_hours=GOES_HOURS,
    start_hour_utc=START_HOUR_UTC,
    end_hour_utc=END_HOUR_UTC,
    overwrite_mask=OVERWRITE_MASK,
    run=RUN_WORKFLOW,
)

print(f"Workflow dates: {config.start_date} to {config.end_date}")
print(f"Bounds: {LON_MIN}, {LAT_MIN}, {LON_MAX}, {LAT_MAX}")
print(f"Daylight UTC window: {START_HOUR_UTC}-{END_HOUR_UTC}")
print(f"Output base: {config.base_dir.resolve()}")

## 3. Check Credentials

GOES is public. OpenTopography and Copernicus/CDS require your own keys.

In [ ]:
wf.validate_credentials(config)

## 4. Run The Workflow

Each step is a one-line function. Progress bars show where the workflow is, and noisy command output is hidden unless a command fails.

In [ ]:
wf.download_goes(config)

In [ ]:
wf.orthorectify(config)

In [ ]:
wf.build_zarr(config)

In [ ]:
rgb_paths = wf.build_rgb(config)
rgb_paths

In [ ]:
mask_paths = wf.apply_mask(config)
mask_paths

## 5. Plot RGB And Mask

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

if not RUN_WORKFLOW:
    raise RuntimeError("Set RUN_WORKFLOW=True and run the workflow cells before plotting outputs.")

plot_date = pd.Timestamp(START_DATE)
rgb_path = config.rgb_path(plot_date)
mask_path = config.mask_path(plot_date)

with xr.open_dataset(rgb_path) as rgb_ds, xr.open_dataset(mask_path) as mask_ds:
    plot_time = pd.Timestamp(PLOT_TIME_UTC) if PLOT_TIME_UTC is not None else pd.Timestamp(mask_ds["t"].values[0])
    rgb_frame = rgb_ds.sel(t=plot_time, method="nearest")
    mask_frame = mask_ds.sel(t=plot_time, method="nearest")
    actual_time = pd.Timestamp(mask_frame["t"].values)

    rgb_image = np.stack(
        [rgb_frame["red"].values, rgb_frame["green"].values, rgb_frame["blue"].values],
        axis=-1,
    )
    rgb_image = np.clip(np.nan_to_num(rgb_image, nan=0.0), 0.0, 1.0)
    cloud_mask = mask_frame["cloud_binary"].values
    lon = rgb_ds["longitude"].values
    lat = rgb_ds["latitude"].values
    extent = [float(np.nanmin(lon)), float(np.nanmax(lon)), float(np.nanmin(lat)), float(np.nanmax(lat))]

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
axes[0].imshow(rgb_image, origin="lower", extent=extent, aspect="auto")
axes[0].set_title(f"GOES RGB\n{actual_time:%Y-%m-%d %H:%M UTC}")
axes[0].set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")

im = axes[1].imshow(cloud_mask, origin="lower", extent=extent, aspect="auto", vmin=0, vmax=1, cmap="Blues_r")
axes[1].set_title("ERA5-Temperature-Bin RGB Cloud Mask")
axes[1].set_xlabel("Longitude")
axes[1].set_ylabel("Latitude")
cbar = fig.colorbar(im, ax=axes[1], ticks=[0, 1], shrink=0.8)
cbar.ax.set_yticklabels(["clear", "cloud"])
plt.show()